In [ ]:
# KAGGLE CELL 1
import os

# 1. Install specific stable versions that work flawlessly
!pip install transformers==4.39.3 accelerate==0.27.2 datasets soundfile librosa Cython

# 2. Clone repo and build Cython
!git clone https://github.com/ylacombe/finetune-hf-vits.git
%cd finetune-hf-vits/monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd /kaggle/working

# 3. Download and extract dataset (Kaggle uses /kaggle/working instead of /content)
!gdown cdsfsdfsdf
import zipfile
os.makedirs('/kaggle/working/dataset', exist_ok=True)
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/kaggle/working/dataset')

In [ ]:
# KAGGLE CELL 2
import os
import json
import urllib.request
from transformers import VitsModel, AutoTokenizer

# --- A. BUILD METADATA.JSONL ---
root_dir = "/kaggle/working/dataset"
metadata_lines =[]
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("metadata.txt"):
            txt_path = os.path.join(dirpath, filename)
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split("|")
                    if len(parts) >= 2:
                        wav_filename, text = parts[0], parts[1]
                        full_wav_path = os.path.join(dirpath, wav_filename)
                        if os.path.exists(full_wav_path):
                            rel_wav_path = os.path.relpath(full_wav_path, root_dir)
                            metadata_lines.append({"file_name": rel_wav_path, "text": text})

with open(os.path.join(root_dir, "metadata.jsonl"), "w", encoding="utf-8") as f:
    for entry in metadata_lines:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# --- B. DOWNLOAD MODEL & FIX META'S BUG ---
local_model_path = "/kaggle/working/local_mms_model"
os.makedirs(local_model_path, exist_ok=True)
VitsModel.from_pretrained("facebook/mms-tts-mon").save_pretrained(local_model_path)
AutoTokenizer.from_pretrained("facebook/mms-tts-mon").save_pretrained(local_model_path)

preprocessor_config = {
    "feature_extractor_type": "VitsFeatureExtractor", "feature_size": 80, "hop_length": 256,
    "max_wav_value": 32768.0, "n_fft": 1024, "padding_side": "right", "padding_value": 0.0,
    "return_attention_mask": False, "sampling_rate": 16000
}
with open(os.path.join(local_model_path, "preprocessor_config.json"), "w") as f:
    json.dump(preprocessor_config, f, indent=4)

# --- C. GENERATE TRAINING CONFIG ---
config = {
    "model_name_or_path": "/kaggle/working/local_mms_model",  
    "dataset_name": "/kaggle/working/dataset",    
    "audio_column_name": "audio",              
    "text_column_name": "text",
    "train_split_name": "train",
    "output_dir": "/kaggle/working/mms-mon-finetuned",
    "report_to": "none",                     # Disables W&B prompts safely
    "do_train": True,
    "do_eval": False,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_steps": 200,
    "max_grad_norm": 1.0,
    "fp16": True,
    "weight_duration": 1.0,
    "weight_kl": 1.5,
    "weight_mel": 35.0,
    "weight_disc": 3.0,
    "weight_gen": 1.0,
    "weight_fmaps": 1.0,
    "max_steps": 100000,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "dataloader_num_workers": 2,
    "logging_steps": 10,
    "save_steps": 200,
    "save_total_limit": 5,
    "push_to_hub": False
}
with open("/kaggle/working/finetune-hf-vits/config.json", "w") as f:
    json.dump(config, f, indent=4)

# --- D. FETCH UNCORRUPTED SCRIPT & PATCH TELEMETRY ---
url = "https://raw.githubusercontent.com/ylacombe/finetune-hf-vits/main/run_vits_finetuning.py"
urllib.request.urlretrieve(url, "/kaggle/working/finetune-hf-vits/run_vits_finetuning.py")

with open("/kaggle/working/finetune-hf-vits/run_vits_finetuning.py", "r") as f:
    script = f.read()

script = script.replace("from transformers.utils import send_example_telemetry", "# from transformers.utils import send_example_telemetry")
script = script.replace("send_example_telemetry(", "# send_example_telemetry(")

with open("/kaggle/working/finetune-hf-vits/run_vits_finetuning.py", "w") as f:
    f.write(script)

print("✅ Kaggle Setup Complete! Ready for Multi-GPU Training.")

# Resampling Audio

In [ ]:
import os
import librosa
import soundfile as sf
import numpy as np
from tqdm.notebook import tqdm

raw_dir = "./dataset/dataset-raw"

print(f"🎧 Safely downsampling and normalizing audio in {raw_dir} to 16,000 Hz Mono...")

wav_files = [f for f in os.listdir(raw_dir) if f.endswith(".wav")]

for wav_file in tqdm(wav_files):
    file_path = os.path.join(raw_dir, wav_file)
    
    # 1. Load the audio (downsamples to 16kHz Mono using high-quality filter)
    audio, sr = librosa.load(file_path, sr=16000, mono=True)
    
    # 2. Peak Normalization: Boost the volume to the maximum safe level 
    # This prevents the 16kHz audio from sounding weak, dull, or quiet
    audio = librosa.util.normalize(audio)
    
    # 3. Overwrite the original file with the clean, loud 16kHz version
    sf.write(file_path, audio, 16000, subtype='PCM_16')

print("✅ All audio safely converted, normalized, and ready for training!")

In [ ]:
# KAGGLE CELL 3
!rm -rf /kaggle/working/mms-mon-finetuned

!accelerate launch \
    --multi_gpu \
    --num_processes=2 \
    --num_machines=1 \
    --mixed_precision="no" \
    --dynamo_backend="no" \
    /kaggle/working/finetune-hf-vits/run_vits_finetuning.py /kaggle/working/finetune-hf-vits/config.json

# Testing

In [ ]:
from transformers import VitsModel, AutoTokenizer, VitsConfig
from safetensors.torch import load_file
import torch
import torch.nn.functional as F
import numpy as np
from IPython.display import Audio, display
import os

# Auto-find latest checkpoint
output_dir = "./mms-mon-finetuned"
checkpoints = sorted(
    [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
checkpoint_path = os.path.join(output_dir, checkpoints[-1])
config_path = output_dir
print(f"🔊 Running inference on: {checkpoint_path}")

# Load raw weights
raw = load_file(f"{checkpoint_path}/model.safetensors")
if os.path.exists(f"{checkpoint_path}/model_1.safetensors"):
    raw.update(load_file(f"{checkpoint_path}/model_1.safetensors"))

# Convert weight_g + weight_v → weight
converted = {}
for key in raw:
    if key.endswith(".weight_g"):
        base = key[:-len(".weight_g")]
        key_v = base + ".weight_v"
        if key_v in raw:
            g = raw[key]
            v = raw[key_v]
            v_norm = F.normalize(v.view(v.shape[0], -1), dim=1).view_as(v)
            converted[base + ".weight"] = g * v_norm
    elif key.endswith(".weight_v"):
        continue
    else:
        converted[key] = raw[key]

print(f"✅ Converted {len([k for k in raw if k.endswith('.weight_g')])} weight_norm pairs")
print(f"✅ Final state dict has {len(converted)} keys")

config = VitsConfig.from_pretrained(config_path)
model = VitsModel(config)
missing, unexpected = model.load_state_dict(converted, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
model.eval()

tokenizer = AutoTokenizer.from_pretrained(config_path)
text = "Өнөөдөр гадаа хасах хорин хэм хүйтэн байна."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

waveform_np = output.squeeze().cpu().numpy()
print("Max amplitude:", np.abs(waveform_np).max())
print("NaN present:", np.isnan(waveform_np).any())

display(Audio(waveform_np, rate=model.config.sampling_rate))

# Resume

In [ ]:
import os
import json

output_dir = "./mms-mon-finetuned"
checkpoints = sorted(
    [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
latest = os.path.join(output_dir, checkpoints[-1])
print(f"▶️ Resuming from: {latest}")

# Inject resume path directly into config.json
config_path = "finetune-hf-vits/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

config["resume_from_checkpoint"] = latest

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

# Launch normally — no extra CLI args needed
!WANDB_DISABLED=true accelerate launch \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=fp16 \
    --dynamo_backend=no \
    ./finetune-hf-vits/run_vits_finetuning.py \
    ./finetune-hf-vits/config.json